In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
import pandas as pd
from f1.data import ROOT

df = pd.read_parquet(ROOT / "data" / "processed" / "laps.parquet")

print(df.shape)
print(df.groupby("Round")["Event"].first())      # which race is each round?
print(df.groupby("Round").agg(laps=("LapNumber", "max"), drivers=("Driver", "nunique")))
print(df["Compound"].value_counts(dropna=False))
print(df.groupby("Round")["Rainfall"].mean().round(2))
print(df.isna().mean().round(3))

(24420, 21)
Round
1           Bahrain Grand Prix
2     Saudi Arabian Grand Prix
3        Australian Grand Prix
4        Azerbaijan Grand Prix
5             Miami Grand Prix
6            Monaco Grand Prix
7           Spanish Grand Prix
8          Canadian Grand Prix
9          Austrian Grand Prix
10          British Grand Prix
11        Hungarian Grand Prix
12          Belgian Grand Prix
13            Dutch Grand Prix
14          Italian Grand Prix
15        Singapore Grand Prix
16         Japanese Grand Prix
17            Qatar Grand Prix
18    United States Grand Prix
19      Mexico City Grand Prix
20        São Paulo Grand Prix
21        Las Vegas Grand Prix
22        Abu Dhabi Grand Prix
Name: Event, dtype: object
       laps  drivers
Round               
1      57.0       20
2      50.0       20
3      58.0       20
4      51.0       20
5      57.0       20
6      78.0       20
7      66.0       20
8      70.0       20
9      71.0       20
10     52.0       20
11     70.0       20


In [11]:
import fastf1
s = fastf1.get_session(2023, 14, "R")
s.load(laps=False, telemetry=False, weather=False, messages=False)
print(set(s.results["Abbreviation"]) - set(df.query("Round == 14")["Driver"]))

core           INFO 	Loading data for Italian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '44', '23', '4', '14', '77', '40', '81', '2', '24', '10', '18', '27', '20', '31', '22']


{'TSU'}


In [12]:
print((df["Compound"] == "None").sum())
print(df[df["Compound"] == "None"][["Round", "Driver", "LapNumber", "TyreLife"]].head(10))

35
      Round Driver  LapNumber  TyreLife
8529      8    TSU       36.0       NaN
8530      8    TSU       37.0       NaN
8531      8    TSU       38.0       NaN
8532      8    TSU       39.0       NaN
8533      8    TSU       40.0       NaN
8534      8    TSU       41.0       NaN
8535      8    TSU       42.0       NaN
8536      8    TSU       43.0       NaN
8537      8    TSU       44.0       NaN
8538      8    TSU       45.0       NaN


In [13]:
wet = df[df["Compound"].isin(["INTERMEDIATE", "WET"])]
print(wet.groupby("Round").size())

Round
6     444
13    328
dtype: int64


In [6]:
from f1.data import load_race_laps

df = load_race_laps(2023, 16)   # Monza was round 16 in 2023; verify via df["Event"]
print(df.shape)
print(df.dtypes)
print(df.isna().mean())
df.head()

core           INFO 	Loading data for Japanese Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 1 completed the race distance 00:00.076000 before the recorded end of the session.
req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '16', '44', '55', '63', '14', '31', '10', '40', '22', '24', '27', '20', '23', '2', '18', '11', '77']


(880, 21)
Driver          object
Team            object
LapNumber      float64
LapTime        float64
Stint          float64
Compound        object
TyreLife       float64
FreshTyre         bool
Position       float64
TrackStatus     object
IsAccurate        bool
IsPitInLap        bool
IsPitOutLap       bool
AirTemp        float64
TrackTemp      float64
Humidity       float64
WindSpeed      float64
Rainfall          bool
Year             int64
Round            int64
Event           object
dtype: object
Driver         0.000000
Team           0.000000
LapNumber      0.000000
LapTime        0.069318
Stint          0.000000
Compound       0.000000
TyreLife       0.000000
FreshTyre      0.000000
Position       0.000000
TrackStatus    0.000000
IsAccurate     0.000000
IsPitInLap     0.000000
IsPitOutLap    0.000000
AirTemp        0.000000
TrackTemp      0.000000
Humidity       0.000000
WindSpeed      0.000000
Rainfall       0.000000
Year           0.000000
Round          0.000000
Event        

,Driver,Team,LapNumber,LapTime,Stint,Compound,TyreLife,FreshTyre,Position,TrackStatus,...,IsPitInLap,IsPitOutLap,AirTemp,TrackTemp,Humidity,WindSpeed,Rainfall,Year,Round,Event
0,VER,Red Bull Racing,1.0,120.179,1.0,MEDIUM,1.0,True,1.0,124,...,False,False,27.0,44.0,42.0,2.5,False,2023,16,Japanese Grand Prix
1,VER,Red Bull Racing,2.0,NaN,1.0,MEDIUM,2.0,True,1.0,4,...,False,False,27.0,43.9,41.0,1.7,False,2023,16,Japanese Grand Prix
2,VER,Red Bull Racing,3.0,NaN,1.0,MEDIUM,3.0,True,1.0,4,...,False,False,27.1,43.5,42.0,1.5,False,2023,16,Japanese Grand Prix
3,VER,Red Bull Racing,4.0,NaN,1.0,MEDIUM,4.0,True,1.0,41,...,False,False,27.2,43.7,41.0,2.5,False,2023,16,Japanese Grand Prix
4,VER,Red Bull Racing,5.0,96.748,1.0,MEDIUM,5.0,True,1.0,1,...,False,False,27.2,42.6,43.0,2.0,False,2023,16,Japanese Grand Prix


In [8]:
df["Event"].iloc[0]
df[df["LapTime"].isna()].groupby("LapNumber").size().head(10)   # NaNs per lap number
df[df["LapTime"].isna()]["TrackStatus"].value_counts().head()

TrackStatus
4      38
41     20
1       2
167     1
Name: count, dtype: int64